# 2단계 (참고): 445명 모델 전체의 test 성능

| 순서 | 노트북 | 내용 |
|---|---|---|
| 0 | `ml_0_dataset` | 학습 데이터셋 결정 (705명 vs 원래 445명 출신 → 445명). class weight를 Optuna로 함께 튜닝 (결과 `kfold_logloss`) |
| 1 | `ml_1_main` | class weight 고정으로 학습하고 모델 비교 (결과 `kfold_logloss_balanced`). 6번에서 최종 모델 1개 선택 |
| 2 | `ml_2_test_all_models` | 참고: 0·1의 445명 모델 전체 test 성능 (모델 선택에 쓰지 않음) |
| 3 | `ml_3_final_test` | 최종 모델 1개 재학습과 test 1회 평가 (최종 모델을 고른 뒤 실행) |

**이 결과는 모델 선택에 쓰지 않습니다.** 최종 모델은 교차검증 결과로 고르고(`ml_1_main`), 이 노트북은 두 학습 방식의 모든 모델이 test 89명(SCD 10명)에서 어떻게 나오는지 기록만 합니다.

| 결과 폴더 | class weight | 하이퍼파라미터 |
|---|---|---|
| `kfold_logloss` | 고정하지 않음 (트리 모델은 Optuna가 함께 탐색) | 트리: Optuna 100회, LR: C 9개 비교 |
| `kfold_logloss_balanced` | 학습 데이터의 SCD 비율로 고정 | 트리: Optuna 100회, LR: C 9개 비교 |

각 모델은 교차검증에서 고른 설정 그대로(재튜닝 없음) 원래 445명 출신 356명 전체로 다시 학습하고, test 89명을 threshold 0.5, 확률 보정 없이 평가합니다.

두 폴더는 class weight가 달라 예측확률의 크기가 다릅니다. 폴더 간 비교는 AUROC·AP로 하고, Brier·민감도·특이도는 폴더 안에서만 비교합니다.

## 0. 준비

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# 저장소 루트를 찾아 import 경로에 추가합니다 (노트북을 어느 폴더에서 열어도 동작).
repo = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Members/mingyu/model/ML/kfold").is_dir()
)
sys.path.insert(0, str(repo))

from Members.mingyu.model.ML.kfold import (  # noqa: E402
    Settings,
    evaluate_on_test,
    fit_final,
    list_results,
    load_cohort,
)  # noqa: E402

RUNS = {  # 결과 폴더 -> 교차검증 때의 class weight 설정
    "kfold_logloss": None,
    "kfold_logloss_balanced": "balanced",
}
TRAIN_FILTER = "only445"

cohort = load_cohort(Settings())
print(
    f"학습 환자: 원래 445명 출신 {cohort.original.sum()}명 (SCD {cohort.y[cohort.original].sum()}) | "
    f"test {len(cohort.test_ids)}명 (SCD {cohort.y_test.sum()})"
)

## 1. 평가할 모델

In [ ]:
targets = {
    tag: [
        name
        for name in list_results(Settings(class_weight=weight, run_tag=tag))
        if name.endswith(f"_{TRAIN_FILTER}")
    ]
    for tag, weight in RUNS.items()
}
for tag, names in targets.items():
    print(f"{tag}: {len(names)}개")
    for name in names:
        print("  ", name)

## 2. 재학습과 test 평가

모델 파일: `<결과 폴더>/<이름>/final_model.joblib`, test 결과: `<결과 폴더>/test/<시각>_<이름>/` (`metrics.json`, `test_predictions.csv`)

`evaluate_on_test`는 기본적으로 test 평가를 한 번만 허용합니다. 여기서는 모든 모델을 기록하는 것이 목적이라 `allow_retest=True`로 실행합니다.

In [ ]:
rows = {}  # (결과 폴더, 방법) -> test 지표


def test_run(tag: str) -> None:
    """한 결과 폴더의 모델을 모두 재학습하고 test 평가합니다."""
    settings = Settings(class_weight=RUNS[tag], run_tag=tag)
    for name in targets[tag]:
        members = fit_final(
            settings, cohort, name
        )  # info.json의 하이퍼파라미터와 class weight 그대로
        rows[(tag, name)] = evaluate_on_test(settings, cohort, name, members, allow_retest=True)

### 2-1. `kfold_logloss`: class weight도 Optuna가 함께 탐색

In [ ]:
test_run("kfold_logloss")

### 2-2. `kfold_logloss_balanced`: class weight를 학습 데이터의 SCD 비율로 고정

In [ ]:
test_run("kfold_logloss_balanced")

## 3. 결과 표

저장: `results/ml_kfold/test_only445_all.csv`

In [ ]:
test_table = pd.DataFrame(rows).T.rename_axis(["결과 폴더", "방법"])
test_table = test_table.sort_values(["결과 폴더", "auroc"], ascending=[True, False])
path = Settings().results_root / f"test_{TRAIN_FILTER}_all.csv"
test_table.to_csv(path)
print("저장:", path)
test_table